# INSURED_VALUE != 0

In [2]:
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules

df = pd.read_csv('insr_>0.csv')

categorical_cols = ['USAGE', 'TYPE_VEHICLE', 'MAKE', 'INSR_TYPE']

df_encoded = df[categorical_cols].copy()
df_encoded['INSR_TYPE'] = df_encoded['INSR_TYPE'].apply(lambda x: str(x))

df_binary = pd.get_dummies(df_encoded)

frequent_itemsets = apriori(df_binary, min_support=0.01, use_colnames=True)
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.5)

print(f"Found {len(rules)} rules")

FileNotFoundError: [Errno 2] No such file or directory: 'insr_>0.csv'

В результате просмотра этих правил были выбраны следующие:

Rule 1:
  IF MAKE_YAMAHA
  THEN TYPE_VEHICLE_Motor-cycle
  Confidence: 100.00% | Lift: 14.35

Rule 2:
  IF MAKE_CALABRESE
  THEN TYPE_VEHICLE_Trailers and semitrailers
  Confidence: 100.00% | Lift: 14.31

Rule 3:
  IF MAKE_FORD, USAGE_Own Goods
  THEN TYPE_VEHICLE_Pick-up
  Confidence: 100.00% | Lift: 4.21

Rule 4:
  IF MAKE_BAJAJ
  THEN TYPE_VEHICLE_Motor-cycle
  Confidence: 99.74% | Lift: 14.31

Rule 5:
  IF USAGE_Fare Paying Passengers
  THEN INSR_TYPE_1202
  Confidence: 99.8631 | Lift: 1.516027

Rule 6:
  IF USAGE_General Cartage
  THEN INSR_TYPE_1202
  Confidence: 99.7392 | Lift: 1.514146

# Выводы

правила со 100% уверенностью (например: 1, 2) будут использоваться для проверки качества тестовых данных.

правила 4, 5, 6 будут использоваться для очистки обучающей выборки и проверки качества тестов.

правило 3 вызывает сомнения (FORD производит не только пикапы).

In [3]:
df[df['MAKE'] == 'FORD']

,SEX,INSR_TYPE,INSURED_VALUE,PREMIUM,OBJECT_ID,PROD_YEAR,SEATS_NUM,TYPE_VEHICLE,CCM_TON,MAKE,USAGE,DURATION,OLD_MAKE,INSR_ZERO
19,0,1202,131737.0,77.550,5000017912,2002.0,4.0,Pick-up,2892.0,FORD,Own Goods,1,FORD,False
20,0,1202,131737.0,2193.569,5000017912,2002.0,4.0,Pick-up,2892.0,FORD,Own Goods,12,FORD,False
21,0,1202,131737.0,2364.770,5000017912,2002.0,4.0,Pick-up,2892.0,FORD,Own Goods,12,FORD,False
22,0,1202,131737.0,2364.770,5000017912,2002.0,4.0,Pick-up,2892.0,FORD,Own Goods,12,FORD,False
170,0,1202,131737.0,77.550,5000018023,2002.0,4.0,Pick-up,2892.0,FORD,Own Goods,1,FORD,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
150301,0,1202,690000.0,6379.750,5000623450,2012.0,2.0,Pick-up,2400.0,FORD,Own Goods,12,FORD,False
150310,0,1202,690000.0,13646.500,5000623484,2011.0,13.0,Bus,2400.0,FORD,Own service,12,FORD,False
150722,0,1202,1200000.0,5376.050,5000629405,2011.0,4.0,Pick-up,2499.0,FORD,Own Goods,12,FORD,False
150755,0,1202,1600000.0,11626.500,5000629908,2013.0,4.0,Pick-up,2200.0,FORD,Own Goods,12,FORD,False


# INSURED_VALUE == 0

In [9]:
def parse_rule_items(items, prefix_sep):
        result = {}
        for item in items:
            col, val = item.split(prefix_sep)
            result[col] = int(val) if col == 'INSR_TYPE' else val
        return result

In [10]:
df = pd.read_csv('insr_=0.csv')

categorical_cols = ['USAGE', 'TYPE_VEHICLE', 'MAKE', 'INSR_TYPE']

df_encoded = df[categorical_cols].copy()
df_encoded['INSR_TYPE'] = df_encoded['INSR_TYPE'].apply(lambda x: str(x))

# Create one-hot encoded matrix
ps = '_!_'
df_binary = pd.get_dummies(df_encoded, prefix_sep=ps)

frequent_itemsets = apriori(df_binary, min_support=0.01, use_colnames=True)
print(frequent_itemsets)
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.5, return_metrics=['confidence'])


rules['antecedents'] = rules['antecedents'].apply(lambda x: parse_rule_items(x, ps))
rules['consequents'] = rules['consequents'].apply(lambda x: parse_rule_items(x, ps))

print(f"Found {len(rules)} rules")

      support                                           itemsets
0    0.118647               frozenset({USAGE_!_General Cartage})
1    0.137647                       frozenset({USAGE_!_Learnes})
2    0.015146                        frozenset({USAGE_!_Others})
3    0.225720                     frozenset({USAGE_!_Own Goods})
4    0.043157                   frozenset({USAGE_!_Own service})
..        ...                                                ...
202  0.014069  frozenset({TYPE_VEHICLE_!_Motor-cycle, MAKE_!_...
203  0.031992  frozenset({TYPE_VEHICLE_!_Station Wagones, USA...
204  0.029545  frozenset({MAKE_!_LADA, INSR_TYPE_!_1202, TYPE...
205  0.047915  frozenset({MAKE_!_TOYOTA, INSR_TYPE_!_1202, TY...
206  0.058498  frozenset({TYPE_VEHICLE_!_Motor-cycle, MAKE_!_...

[207 rows x 2 columns]
Found 360 rules


In [17]:
rules[rules['confidence'] == 1.0]

,antecedents,consequents,confidence
28,{'USAGE': 'Special Construction'},{'INSR_TYPE': 1202},1.0
39,{'MAKE': 'TVS'},{'TYPE_VEHICLE': 'Motor-cycle'},1.0
40,{'MAKE': 'YAMAHA'},{'TYPE_VEHICLE': 'Motor-cycle'},1.0
48,{'TYPE_VEHICLE': 'Tanker'},{'INSR_TYPE': 1202},1.0
101,"{'MAKE': 'BAJAJ', 'USAGE': 'Others'}",{'TYPE_VEHICLE': 'Motor-cycle'},1.0
134,"{'MAKE': 'OPEL', 'USAGE': 'Own Goods'}",{'INSR_TYPE': 1202},1.0
158,"{'USAGE': 'Private', 'MAKE': 'BAJAJ'}",{'TYPE_VEHICLE': 'Motor-cycle'},1.0
160,"{'MAKE': 'YAMAHA', 'USAGE': 'Private'}",{'TYPE_VEHICLE': 'Motor-cycle'},1.0
184,"{'TYPE_VEHICLE': 'Special construction', 'USAG...",{'INSR_TYPE': 1202},1.0
188,"{'MAKE': 'LADA', 'USAGE': 'Taxi'}",{'TYPE_VEHICLE': 'Automobile'},1.0


In [11]:
rules.iloc[320]['antecedents']

frozenset({'INSR_TYPE_1201', 'MAKE_VOLKSWAGEN', 'TYPE_VEHICLE_Automobile'})

Rule 1:
  IF MAKE_LADA, USAGE_Taxi, INSR_TYPE_1202
  THEN TYPE_VEHICLE_Automobile
  Confidence: 1.000000 | Lift: 4.333165

Rule 2:
  IF INSR_TYPE_1201, MAKE_VOLKSWAGEN, TYPE_VEHICLE_Automobile
  THEN USAGE_Private
  Confidence: 0.99916 | Lift: 3.554573

Rule 3:
  IF USAGE_Taxi, MAKE_BAJAJ
  THEN frozenset({INSR_TYPE_1202})
  Confidence: 0.999838 | Lift: 1.398611

Rule 4:
  IF USAGE_Special Construction
  THEN INSR_TYPE_1202
  Confidence: 1.0 | Lift: 1.398837

Rule 5:
  IF MAKE_TVS
  THEN TYPE_VEHICLE_Motor-cycle
  Confidence: 1.0 | Lift: 4.378108

Применение аналогично

Пример конфига

In [ ]:
rules_zero = [
    {
        'antecedents': {'MAKE': 'LADA', 'USAGE': 'Taxi', 'INSR_TYPE': 1202},
        'consequent': {'TYPE_VEHICLE': 'Automobile'},
        'confidence': 1.000000
    },
    {
        'antecedents': {'MAKE': 'VOLKSWAGEN', 'TYPE_VEHICLE': 'Automobile', 'INSR_TYPE': 1201},
        'consequent': {'USAGE': 'Private'},
        'confidence': 0.99916
    },
    {
        'antecedents': {'MAKE': 'BAJAJ', 'USAGE': 'Taxi'},
        'consequent': {'INSR_TYPE': 1202},
        'confidence': 0.999838
    },
    {
        'antecedents': {'USAGE': 'Special Construction'},
        'consequent': {'INSR_TYPE': 1202},
        'confidence': 1.000000
    },
    {
        'antecedents': {'MAKE': 'TVS'},
        'consequent': {'TYPE_VEHICLE': 'Motor-cycle'},
        'confidence': 1.000000
    }
]

rules_else = [
    {
        'antecedents': {'MAKE': 'YAMAHA'},
        'consequent': {'TYPE_VEHICLE': 'Motor-cycle'},
        'confidence': 1.000000
    },
    {
        'antecedents': {'MAKE': 'CALABRESE'},
        'consequent': {'TYPE_VEHICLE': 'Trailers and semitrailers'},
        'confidence': 1.0
    },
    {
        'antecedents': {'MAKE': 'BAJAJ'},
        'consequent': {'TYPE_VEHICLE': 'Motor-cycle'},
        'confidence': 0.9974
    },
    {
        'antecedents': {'USAGE': 'Fare Paying Passengers'},
        'consequent': {'INSR_TYPE': 1202},
        'confidence': 0.998631
    },
    {
        'antecedents': {'USAGE': 'General Cartage'},
        'consequent': {'INSR_TYPE': 1202},
        'confidence': 0.9973
    }
]

apriori_config = {"INSR_ZERO": rules_zero, "ELSE": rules_else}

import json
with open('apriori_rules.json', 'w') as f:
    json.dump(apriori_config, f)